# The vehicle detector, step by step

A camera watching a kerb has one job: say which stretches of it are free. That reduces to
finding the vehicles, and this notebook trains the detector that does it, exports it to
ONNX, and proves the export did not change it.

Running the whole notebook is equivalent to:

```
pf detect dataset
pf detect train
pf detect export
```

Every cell calls the same modules, so nothing here is a parallel implementation.

In [ ]:
from parkfit.numeric import limit_numeric_threads

limit_numeric_threads()

# Before parkfit.ml.viz is imported. viz falls back to a headless backend when no
# IPython session has claimed one, which is right for the CLI and wrong here.
%matplotlib inline

import logging
from pathlib import Path

import numpy as np

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("parkfit.ml.datasets").setLevel(logging.WARNING)

FIGURES = Path("../data/figures")
DATASET = Path("../data/detector")
FIGURES.mkdir(parents=True, exist_ok=True)

from parkfit.ml import viz
from parkfit.ml.datasets import scenes

print("ready")

## 1. Ground truth you can trust

The dataset is rendered, and that is the point. A real camera gives you pixels and no
answer, so any accuracy number measured against real footage is really a number about
whoever drew the boxes. Here the renderer placed the cars, so it knows exactly where they
are and exactly how long each gap is.

**The bug this step already caught.** The renderer draws a kerb far longer than the camera
can see, and `Scene.detections()` reports every vehicle on it, in frame or not. At the
default 18 m mount only about 13 m of a 40 m kerb is visible, and four of six boxes in the
first validation scene sat entirely outside the image. The builder now sizes the kerb to
the visible span and clips what remains.

In [ ]:
from parkfit.ml.synthetic.scene import CameraModel

camera = CameraModel()
print(
    f"{camera.width_px}x{camera.height_px}, focal {camera.focal_px}px, "
    f"tilt {camera.tilt_deg} deg, mounted {camera.height_m} m up\n"
)

for offset in (14, 18, 22, 26, 30):
    visible = scenes.visible_kerb_m(camera, offset)
    print(f"kerb {offset:2d} m away -> {visible:5.1f} m of kerb in frame")

In [ ]:
report = scenes.build(DATASET, train_scenes=600, val_scenes=150, seed=7)
print(report.describe())
print("scenes per lighting condition:", report.per_condition)

train_split, val_split = scenes.load(DATASET)
images = val_split.images()
sample = np.stack([images[val_split.index_offset + i] for i in range(6)])
viz.grid_of_scenes(sample, val_split.labels[:6], out_dir=FIGURES);

## 2. What the model is actually asked to predict

CenterNet: a per-class heatmap whose peaks are object centres, plus a box size and a
sub-pixel offset at each peak. Chosen over an anchor-based detector for one practical
reason above all others, which is that decoding needs no anchor bookkeeping and no
suppression inside the graph. On the C++ side the whole decoder is a 3x3 local-maximum
test.

Worth looking at the target at least once. This is the step where an indexing mistake is
obvious to the eye and completely invisible in the loss, which will happily descend while
training on peaks in the wrong places.

In [ ]:
heatmap, size, offset, mask = scenes.encode_targets(val_split.labels[0])
print(f"heatmap {heatmap.shape}, size {size.shape}, offset {offset.shape}")
print(f"{int(mask.sum())} centre cells, peak value {heatmap.max():.3f}")

viz.target_heatmap(heatmap, out_dir=FIGURES);

The splat radius is derived from the object's size rather than fixed. A fixed radius would
smear a distant motorcycle's peak across its neighbours and give a truck a peak too sharp
to ever hit.

In [ ]:
viz.gaussian_radius_curve(out_dir=FIGURES);

### The round trip that everything is measured through

Encode the truth, decode it back, and check nothing moved. Every accuracy number in this
notebook is measured *through* those two functions, so if they disagree the model can train
perfectly and score badly for reasons no amount of staring at the loss curve will explain.

In [ ]:
from parkfit.ml.train import detector

recovered = detector.decode(heatmap, size, offset, threshold=0.99)
tp, fp, fn, mae = detector.match(recovered, val_split.labels[0])
print(f"truth {len(val_split.labels[0])}, decoded {len(recovered)}")
print(f"true positives {tp}, false positives {fp}, missed {fn}, corner error {mae:.6f} px")
assert fn == 0 and fp == 0, "encode/decode is lossy, stop here"

## 3. Training

Roughly 320k parameters. The job is not open-world detection; it is finding vehicles
against a road in a fixed camera. A ResNet would train slower, export bigger and measure
nothing extra.

Three losses, and only one is dense. The heatmap uses the CornerNet focal variant, which
down-weights the enormous background rather than letting it drown the few positive cells.
Size and offset are L1 evaluated **only** at true centre cells, because "what size is the
object at this empty patch of road" has no answer to regress toward.

This takes a few minutes on CPU. To skip it, load the weights from a previous run instead.

In [ ]:
train_report = detector.train(DATASET, epochs=26, batch_size=8, threads=4)
print(train_report.describe())

In [ ]:
viz.training_curves(train_report.history, out_dir=FIGURES);

The three panels are separate on purpose. The size term is an L1 in pixels and runs an
order of magnitude above the other two; sharing one axis would flatten the heatmap and
offset curves into a line along the bottom, and a second y-axis would make the two look
comparable when they are not.

In [ ]:
viz.condition_f1(train_report.per_condition, out_dir=FIGURES);

Night is the weakest condition, which is both expected and what the numbers say. The chart
is sorted worst-first because that is the useful question.

## 4. Looking at the predictions

Numbers are not enough. Truth is dashed, prediction is solid, so the two stay separable in
print and for a colourblind reader without reading a legend.

In [ ]:
import torch

model = detector.build_model()
model.load_state_dict(torch.load(detector.DEFAULT_WEIGHTS, map_location="cpu"))
model.eval()

index = 0
frame = images[val_split.index_offset + index]
batch = np.transpose(frame.astype(np.float32) / 255.0, (2, 0, 1))[None]

with torch.no_grad():
    heat, size_out, offset_out = model(torch.from_numpy(batch))

predicted = detector.decode(heat[0].numpy(), size_out[0].numpy(), offset_out[0].numpy())
viz.scene_with_boxes(
    frame,
    truth=val_split.labels[index],
    predicted=predicted,
    title=f"{val_split.conditions[index]}: {len(predicted)} detected, "
    f"{len(val_split.labels[index])} present",
    out_dir=FIGURES,
    name="prediction",
);

## 5. Export, and proving the export did not change anything

An export is a translation between two implementations, and translations go wrong quietly.
A fused batch-norm with the wrong epsilon, an interpolate that resolves to a different
rounding rule: none of these raise, and every one shifts boxes by a few pixels in a way
that only shows up as a mysteriously worse accuracy number weeks later.

So exporting is not finished when the file is written. The same frames run through PyTorch
and through ONNX Runtime, comparing raw tensors **and** decoded boxes. The tensor check
catches numerical drift; the box check catches a small tensor difference landing either
side of a decision boundary and moving a detection.

Tolerance is relative **per output**. The heatmap and offset live in [0, 1] while the size
head emits pixels running to a couple of hundred, so one absolute threshold is either far
too tight for size or meaningless for the other two. The first version used 1e-4 absolute
and reported a mismatch on a size difference of 2.9e-4, which is one and a half parts per
million, while the decoded boxes were identical to the last decimal place.

In [ ]:
from parkfit.ml.export import onnx as onnx_export

export_report = onnx_export.export(dataset_root=DATASET)
print(export_report.describe())
assert export_report.agrees, "the export does not reproduce the model, do not ship it"

## 6. The C++ side

The exported graph is run by `pf_cv_worker` through ONNX Runtime's C API, loaded with
`LoadLibrary` at startup rather than linked, so the worker builds and runs on a machine
that has never installed it.

The two decoders, Python here and C++ in `cpp/vision/src/onnx_detector.cpp`, are tested
separately against hand-built tensors so neither can drift into agreeing on something
wrong. Then they are checked against each other on real frames through the real model:

```
pf_cv_worker --replay data/synthetic/scene_ \
             --onnx data/models/detector.onnx \
             --max-frames 3 --verbose
```

Measured: identical detections, scores equal to six decimal places.

| Frame | Python | C++ |
|---|---|---|
| scene_000 | car 0.767944 [973.5268, 394.0856, 1159.3485, 464.1636] | car 0.767944 [973.527, 394.086, 1159.35, 464.164] |
| scene_001 | car 0.533868 [997.3109, 395.0065, 1164.7853, 463.1137] | car 0.533868 [997.311, 395.007, 1164.79, 463.114] |

That covers preprocessing (nearest-neighbour resize, channel order, normalisation), the
decode, and the rescale back to frame coordinates.